In [1]:
!cp /content/base_data_pipeline.py /content

cp: '/content/base_data_pipeline.py' and '/content/base_data_pipeline.py' are the same file


In [2]:
import base_data_pipeline

In [3]:
train_texts, test_texts, train_labels, test_labels = base_data_pipeline.get_processed_data()

In [4]:
df = base_data_pipeline.load_dataset()

In [5]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

df.head()


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [8]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [9]:
class SMSDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [10]:
train_dataset = SMSDataset(train_encodings, train_labels)
test_dataset = SMSDataset(test_encodings, test_labels)

In [11]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(torch.cuda.is_available)
print(device)

<function is_available at 0x7b92d037d260>
cuda


In [14]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [15]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import copy

num_epochs = 10
patience = 2

best_f1 = 0
epochs_no_improve = 0
best_model_state = None

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):

        optimizer.zero_grad()

        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)

        loss = outputs.loss
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Training Loss: {avg_loss}")

    # evaluation
    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():

        for batch in test_loader:

            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)

            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(batch["labels"].cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predictions, average="binary"
    )

    print("Accuracy:", accuracy)
    print("F1 Score:", f1)

    if f1 > best_f1:

        best_f1 = f1
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0

        print("Improvement detected. Saving model.")

    else:

        epochs_no_improve += 1
        print("No improvement. Patience counter:", epochs_no_improve)

    if epochs_no_improve >= patience:

        print("\nEarly stopping triggered.")
        break


model.load_state_dict(best_model_state)

print("\nBest model restored with F1:", best_f1)

100%|██████████| 279/279 [01:38<00:00,  2.82it/s]



Epoch 1 Training Loss: 0.08050446779430447
Accuracy: 0.968609865470852
F1 Score: 0.8909657320872274
Improvement detected. Saving model.


100%|██████████| 279/279 [01:41<00:00,  2.76it/s]



Epoch 2 Training Loss: 0.02867433544595502
Accuracy: 0.9901345291479821
F1 Score: 0.962457337883959
Improvement detected. Saving model.


100%|██████████| 279/279 [01:41<00:00,  2.76it/s]



Epoch 3 Training Loss: 0.020412067210512055
Accuracy: 0.9928251121076234
F1 Score: 0.9727891156462585
Improvement detected. Saving model.


100%|██████████| 279/279 [01:41<00:00,  2.76it/s]



Epoch 4 Training Loss: 0.012445245942034748
Accuracy: 0.9919282511210762
F1 Score: 0.9692832764505119
No improvement. Patience counter: 1


100%|██████████| 279/279 [01:41<00:00,  2.76it/s]



Epoch 5 Training Loss: 0.001614416371364402
Accuracy: 0.9919282511210762
F1 Score: 0.9692832764505119
No improvement. Patience counter: 2

Early stopping triggered.

Best model restored with F1: 0.9727891156462585


In [16]:
!cp /content/evaluation_utils.py /content

cp: '/content/evaluation_utils.py' and '/content/evaluation_utils.py' are the same file


In [17]:
import evaluation_utils

In [18]:
results = evaluation_utils.evaluate_model(true_labels, predictions)

In [19]:
results

{'Accuracy': 0.9919282511210762,
 'Precision': 0.9861111111111112,
 'Recall': 0.9530201342281879,
 'F1 Score': 0.9692832764505119,
 'Confusion Matrix': array([[964,   2],
        [  7, 142]])}